# Celda 0 — Configuración

In [ ]:
# ════════════════════════════════════
# CELDA 0 — Configuración: Drive y rutas
# ════════════════════════════════════
import os

try:
    # Entorno Google Colab: se monta Drive (como pide el PDF)
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BASE = '/content/drive/MyDrive/SGB/datos/'
    ENTORNO = 'Colab + Drive'
except ImportError:
    # Entorno local (VS Code): carpeta datos/ junto al notebook
    RUTA_BASE = os.path.join(os.getcwd(), 'datos', '')
    ENTORNO = 'Local (VS Code)'

os.makedirs(RUTA_BASE, exist_ok=True)

# Rutas de cada archivo de persistencia
RUTA_USUARIOS   = RUTA_BASE + 'usuarios.txt'
RUTA_LIBROS     = RUTA_BASE + 'libros.txt'
RUTA_CATEGORIAS = RUTA_BASE + 'categorias.txt'
RUTA_PRESTAMOS  = RUTA_BASE + 'prestamos.txt'
RUTA_MULTAS     = RUTA_BASE + 'multas.txt'
RUTA_RESERVAS   = RUTA_BASE + 'reservas.txt'

if os.path.isdir(RUTA_BASE):
    print(f'✔ Entorno: {ENTORNO}')
    print('✔ Carpeta de datos lista en:', RUTA_BASE)
else:
    print('✘ No se pudo crear la carpeta de datos.')

    

# Sección 1 — Modelo: Enums

In [ ]:
# ════════════════════════════════════
# CELDA 1.1 — Enums
# ════════════════════════════════════
from enum import Enum


class TipoUsuarioEnum(Enum):
    """Tipos de usuario de la biblioteca."""
    ESTUDIANTE = 'ESTUDIANTE'
    DOCENTE = 'DOCENTE'
    EXTERNO = 'EXTERNO'


class EstadoPrestamoEnum(Enum):
    """Estados posibles de un préstamo."""
    ACTIVO = 'ACTIVO'
    DEVUELTO = 'DEVUELTO'
    VENCIDO = 'VENCIDO'


class EstadoMultaEnum(Enum):
    """Estados posibles de una multa."""
    PENDIENTE = 'PENDIENTE'
    PAGADA = 'PAGADA'


class EstadoReservaEnum(Enum):
    """Estados posibles de una reserva."""
    ACTIVA = 'ACTIVA'
    CUMPLIDA = 'CUMPLIDA'
    CANCELADA = 'CANCELADA'

# Sección 2 — Repositorios: ArchivoUtil

In [ ]:
# ════════════════════════════════════
# CELDA 2.1 — ArchivoUtil
# ════════════════════════════════════
import os
from datetime import datetime


class ArchivoUtil:
    """Utilidades para leer y escribir archivos de texto (.txt) en Google Drive.

    Cada línea del archivo es un registro. Todos los métodos son estáticos,
    por lo que no hace falta crear un objeto para usarlos.
    """

    @staticmethod
    def existe(ruta):
        """Retorna True si el archivo existe en la ruta indicada."""
        return os.path.isfile(ruta)

    @staticmethod
    def crear_si_no_existe(ruta):
        """Crea un archivo vacío si no existe.

        Retorna True si lo creó y False si ya existía.
        """
        if os.path.isfile(ruta):
            return False
        carpeta = os.path.dirname(ruta)
        if carpeta:
            os.makedirs(carpeta, exist_ok=True)
        with open(ruta, 'a', encoding='utf-8'):
            pass
        return True

    @staticmethod
    def leer_lineas(ruta):
        """Lee todas las líneas no vacías de un archivo.

        Retorna una lista vacía si el archivo no existe.
        """
        try:
            with open(ruta, 'r', encoding='utf-8') as f:
                return [l.strip() for l in f if l.strip()]
        except FileNotFoundError:
            return []

    @staticmethod
    def escribir_linea(ruta, linea):
        """Agrega una línea al final del archivo.

        Lanza ValueError si la línea contiene saltos de línea, porque
        rompería la regla de un registro por línea.
        """
        if '\n' in linea or '\r' in linea:
            raise ValueError('La línea no puede contener saltos de línea.')
        with open(ruta, 'a', encoding='utf-8') as f:
            f.write(linea + '\n')

    @staticmethod
    def reescribir(ruta, lineas):
        """Reemplaza el contenido completo del archivo con las líneas dadas.

        Si la lista está vacía, deja el archivo vacío.
        """
        with open(ruta, 'w', encoding='utf-8') as f:
            if lineas:
                f.write('\n'.join(lineas) + '\n')

    @staticmethod
    def generar_codigo(prefijo):
        """Genera un código único basado en la fecha y hora actuales.

        Ejemplo: generar_codigo('PRE') -> 'PRE-20251015083000123'
        """
        ts = datetime.now().strftime('%Y%m%d%H%M%S%f')[:17]
        return f'{prefijo}-{ts}'

# Verificación P0


In [ ]:
# ════════════════════════════════════
# VERIFICACIÓN P0 — Persistencia en Drive
# ════════════════════════════════════
import re

# 1) Crear los 6 archivos si no existen
rutas = {
    'usuarios.txt': RUTA_USUARIOS,
    'categorias.txt': RUTA_CATEGORIAS,
    'libros.txt': RUTA_LIBROS,
    'prestamos.txt': RUTA_PRESTAMOS,
    'multas.txt': RUTA_MULTAS,
    'reservas.txt': RUTA_RESERVAS,
}
print('--- Archivos de persistencia ---')
for nombre, ruta in rutas.items():
    creado = ArchivoUtil.crear_si_no_existe(ruta)
    estado = 'creado' if creado else 'ya existía'
    print(f'  {nombre:<15} {estado:<11} existe={ArchivoUtil.existe(ruta)}')

# 2) Prueba de escritura y lectura con un archivo temporal
RUTA_PRUEBA = RUTA_BASE + 'prueba_p0.txt'
if ArchivoUtil.existe(RUTA_PRUEBA):
    os.remove(RUTA_PRUEBA)

print('\n--- Prueba de lectura/escritura ---')
assert ArchivoUtil.leer_lineas(RUTA_PRUEBA) == [], 'Archivo inexistente debe dar []'
print('  ✔ Leer archivo inexistente devuelve lista vacía')

ArchivoUtil.escribir_linea(RUTA_PRUEBA, '1098765432|Maria Lopez|ESTUDIANTE|maria.lopez@correo.com|3001234567|ACTIVO')
ArchivoUtil.escribir_linea(RUTA_PRUEBA, '1001234567|Juan Perez|DOCENTE|juan.perez@correo.com|3109876543|ACTIVO')
lineas = ArchivoUtil.leer_lineas(RUTA_PRUEBA)
assert len(lineas) == 2, 'Debían leerse 2 líneas'
assert lineas[0].split('|')[1] == 'Maria Lopez'
print('  ✔ escribir_linea + leer_lineas:', len(lineas), 'registros')

ArchivoUtil.reescribir(RUTA_PRUEBA, [lineas[1]])
assert ArchivoUtil.leer_lineas(RUTA_PRUEBA) == [lineas[1]]
print('  ✔ reescribir: el archivo quedó con 1 registro')

codigo = ArchivoUtil.generar_codigo('PRE')
assert re.fullmatch(r'PRE-\d{17}', codigo), f'Formato inesperado: {codigo}'
print('  ✔ generar_codigo:', codigo)

try:
    ArchivoUtil.escribir_linea(RUTA_PRUEBA, 'línea\ncon salto')
    print('  ✘ Debió lanzar ValueError')
except ValueError:
    print('  ✔ Línea con salto de línea rechazada (ValueError)')

os.remove(RUTA_PRUEBA)
print('\n✔ P0 verificado. Archivos en la carpeta:', sorted(os.listdir(RUTA_BASE)))

# Celdas 1.2 a 1.4 — Clases del modelo (P1)

In [ ]:
# ════════════════════════════════════
# CELDA 1.2 — Clase Usuario
# ════════════════════════════════════
SEPARADOR = '|'
VACIO = 'NINGUNO'


def _validar_campo(valor):
    """Verifica que un campo no contenga el separador '|' ni saltos de línea.

    Retorna el valor convertido a texto. Lanza ValueError si no es válido.
    """
    texto = str(valor)
    if SEPARADOR in texto or '\n' in texto or '\r' in texto:
        raise ValueError(f"El valor '{texto}' no puede contener '|' ni saltos de línea.")
    return texto


class Usuario:
    """Persona que usa la biblioteca (estudiante, docente o externo)."""

    CAMPOS = 6

    def __init__(self, num_documento, nombre, tipo, correo, telefono, estado='ACTIVO'):
        """Crea un usuario.

        tipo debe ser un TipoUsuarioEnum; estado es 'ACTIVO' o 'SUSPENDIDO'.
        """
        self.num_documento = num_documento
        self.nombre = nombre
        self.tipo = tipo
        self.correo = correo
        self.telefono = telefono
        self.estado = estado

    def to_linea(self):
        """Convierte el usuario en una línea de texto para guardar en usuarios.txt."""
        return SEPARADOR.join([
            _validar_campo(self.num_documento),
            _validar_campo(self.nombre),
            self.tipo.value,
            _validar_campo(self.correo),
            _validar_campo(self.telefono),
            _validar_campo(self.estado),
        ])

    @staticmethod
    def from_linea(linea):
        """Crea un Usuario a partir de una línea de usuarios.txt.

        Lanza ValueError si la línea no tiene 6 campos o el tipo no es válido.
        """
        campos = linea.split(SEPARADOR)
        if len(campos) != Usuario.CAMPOS:
            raise ValueError(f'Línea de usuario inválida: {linea!r}')
        doc, nombre, tipo, correo, telefono, estado = campos
        return Usuario(doc, nombre, TipoUsuarioEnum(tipo), correo, telefono, estado)

    def __str__(self):
        """Texto legible del usuario para mostrar en pantalla."""
        return (f'{self.num_documento} | {self.nombre} | {self.tipo.value} | '
                f'{self.correo} | {self.telefono} | {self.estado}')


# ════════════════════════════════════
# CELDA 1.3 — Clase Categoria
# ════════════════════════════════════
class Categoria:
    """Grupo temático de libros (por ejemplo, Literatura o Tecnología)."""

    CAMPOS = 3

    def __init__(self, codigo, nombre, descripcion=None):
        """Crea una categoría. La descripción es opcional (None si no hay)."""
        self.codigo = codigo
        self.nombre = nombre
        self.descripcion = descripcion

    def to_linea(self):
        """Convierte la categoría en una línea para categorias.txt."""
        descripcion = self.descripcion if self.descripcion else VACIO
        return SEPARADOR.join([
            _validar_campo(self.codigo),
            _validar_campo(self.nombre),
            _validar_campo(descripcion),
        ])

    @staticmethod
    def from_linea(linea):
        """Crea una Categoria a partir de una línea de categorias.txt.

        Lanza ValueError si la línea no tiene 3 campos.
        """
        campos = linea.split(SEPARADOR)
        if len(campos) != Categoria.CAMPOS:
            raise ValueError(f'Línea de categoría inválida: {linea!r}')
        codigo, nombre, descripcion = campos
        return Categoria(codigo, nombre, None if descripcion == VACIO else descripcion)

    def __str__(self):
        """Texto legible de la categoría para mostrar en pantalla."""
        return f'{self.codigo} | {self.nombre} | {self.descripcion or "Sin descripción"}'

# ════════════════════════════════════
# CELDA 1.4 — Clase Libro
# ════════════════════════════════════
class Libro:
    """Título del catálogo, con control de ejemplares totales y disponibles."""

    CAMPOS = 7

    def __init__(self, isbn, titulo, autor, cod_categoria, anio, total_ejemplares, disponibles=None):
        """Crea un libro.

        Si no se indica 'disponibles', se asume que todos los ejemplares lo están.
        """
        self.isbn = isbn
        self.titulo = titulo
        self.autor = autor
        self.cod_categoria = cod_categoria
        self.anio = anio
        self.total_ejemplares = total_ejemplares
        self.disponibles = total_ejemplares if disponibles is None else disponibles

    def to_linea(self):
        """Convierte el libro en una línea para libros.txt."""
        return SEPARADOR.join([
            _validar_campo(self.isbn),
            _validar_campo(self.titulo),
            _validar_campo(self.autor),
            _validar_campo(self.cod_categoria),
            str(self.anio),
            str(self.total_ejemplares),
            str(self.disponibles),
        ])

    @staticmethod
    def from_linea(linea):
        """Crea un Libro a partir de una línea de libros.txt.

        Lanza ValueError si la línea no tiene 7 campos o los números no son válidos.
        """
        campos = linea.split(SEPARADOR)
        if len(campos) != Libro.CAMPOS:
            raise ValueError(f'Línea de libro inválida: {linea!r}')
        isbn, titulo, autor, cod_cat, anio, total, disp = campos
        return Libro(isbn, titulo, autor, cod_cat, int(anio), int(total), int(disp))

    def __str__(self):
        """Texto legible del libro para mostrar en pantalla."""
        return (f'{self.isbn} | {self.titulo} | {self.autor} | {self.cod_categoria} | '
                f'{self.anio} | {self.disponibles}/{self.total_ejemplares} disponibles')


# Prueba Bloque A

In [ ]:
# PRUEBA BLOQUE A — to_linea / from_linea
u = Usuario('1098765432', 'Maria Lopez', TipoUsuarioEnum.ESTUDIANTE,
            'maria.lopez@correo.com', '3001234567')
linea_u = u.to_linea()
print(linea_u)
assert linea_u == '1098765432|Maria Lopez|ESTUDIANTE|maria.lopez@correo.com|3001234567|ACTIVO'
u2 = Usuario.from_linea(linea_u)
assert u2.tipo == TipoUsuarioEnum.ESTUDIANTE and u2.nombre == 'Maria Lopez'
print('✔ Usuario: ida y vuelta correcta')

c = Categoria('LIT', 'Literatura')
linea_c = c.to_linea()
print(linea_c)
assert linea_c == 'LIT|Literatura|NINGUNO'
assert Categoria.from_linea(linea_c).descripcion is None
print('✔ Categoria: campo vacío se guarda como NINGUNO')

l = Libro('9789585001234', 'Cien anios de soledad', 'Gabriel Garcia Marquez', 'LIT', 1967, 5, 3)
linea_l = l.to_linea()
print(linea_l)
assert linea_l == '9789585001234|Cien anios de soledad|Gabriel Garcia Marquez|LIT|1967|5|3'
l2 = Libro.from_linea(linea_l)
assert l2.anio == 1967 and l2.disponibles == 3
print('✔ Libro: ida y vuelta correcta')

try:
    Usuario('1', 'Nombre|malo', TipoUsuarioEnum.EXTERNO, 'a@b.com', '300').to_linea()
    print('✘ Debió lanzar ValueError')
except ValueError:
    print("✔ Un campo con '|' es rechazado")

try:
    Usuario.from_linea('solo|tres|campos')
    print('✘ Debió lanzar ValueError')
except ValueError:
    print('✔ Línea con campos incompletos es rechazada')

print('\n✔ Bloque A verificado')

# Sección 2 — Repositorios (P1)

In [ ]:
# ════════════════════════════════════
# CELDA 2.2 — UsuarioRepository
# ════════════════════════════════════
class UsuarioRepository:
    """Guarda y consulta usuarios en usuarios.txt."""

    def __init__(self, ruta):
        """Recibe la ruta del archivo de usuarios (RUTA_USUARIOS)."""
        self.ruta = ruta

    def guardar(self, u):
        """Agrega un usuario nuevo al final del archivo."""
        ArchivoUtil.escribir_linea(self.ruta, u.to_linea())

    def find_by_documento(self, doc):
        """Busca un usuario por número de documento. Retorna Usuario o None."""
        for u in self.find_all():
            if u.num_documento == doc:
                return u
        return None

    def find_all(self):
        """Retorna la lista de todos los usuarios registrados."""
        return [Usuario.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

    def actualizar(self, u):
        """Reemplaza los datos del usuario que tenga el mismo documento.

        Retorna True si lo encontró y actualizó, False si no existe.
        """
        usuarios = self.find_all()
        encontrado = False
        for i, existente in enumerate(usuarios):
            if existente.num_documento == u.num_documento:
                usuarios[i] = u
                encontrado = True
                break
        if encontrado:
            ArchivoUtil.reescribir(self.ruta, [x.to_linea() for x in usuarios])
        return encontrado

# ════════════════════════════════════
# CELDA 2.3 — CategoriaRepository
# ════════════════════════════════════
class CategoriaRepository:
    """Guarda y consulta categorías en categorias.txt."""

    def __init__(self, ruta):
        """Recibe la ruta del archivo de categorías (RUTA_CATEGORIAS)."""
        self.ruta = ruta

    def guardar(self, c):
        """Agrega una categoría nueva al final del archivo."""
        ArchivoUtil.escribir_linea(self.ruta, c.to_linea())

    def find_by_codigo(self, cod):
        """Busca una categoría por código. Retorna Categoria o None."""
        for c in self.find_all():
            if c.codigo == cod:
                return c
        return None

    def find_all(self):
        """Retorna la lista de todas las categorías."""
        return [Categoria.from_linea(l) for l in ArchivoUtil.leer_lineas(self.ruta)]

# ════════════════════════════════════
# CELDA 2.4 — LibroRepository
# ════════════════════════════════════
class LibroRepository:
    """Guarda y consulta libros en libros.txt."""

    def __init__(self, ruta):
        """Recibe la ruta del archivo de libros (RUTA_LIBROS)."""
        self.ruta = ruta

    def guardar(self, l):
        """Agrega un libro nuevo al final del archivo."""
        ArchivoUtil.escribir_linea(self.ruta, l.to_linea())

    def find_all(self):
        """Retorna la lista de todos los libros del catálogo."""
        return [Libro.from_linea(x) for x in ArchivoUtil.leer_lineas(self.ruta)]

    def find_by_isbn(self, isbn):
        """Busca un libro por ISBN. Retorna Libro o None."""
        for l in self.find_all():
            if l.isbn == isbn:
                return l
        return None

    def find_by_titulo(self, txt):
        """Retorna los libros cuyo título contiene el texto (sin distinguir mayúsculas)."""
        txt = txt.lower()
        return [l for l in self.find_all() if txt in l.titulo.lower()]

    def find_by_categoria(self, cod):
        """Retorna los libros de la categoría con ese código."""
        return [l for l in self.find_all() if l.cod_categoria == cod]

    def actualizar(self, l):
        """Reemplaza los datos del libro que tenga el mismo ISBN.

        Retorna True si lo encontró y actualizó, False si no existe.
        """
        libros = self.find_all()
        encontrado = False
        for i, existente in enumerate(libros):
            if existente.isbn == l.isbn:
                libros[i] = l
                encontrado = True
                break
        if encontrado:
            ArchivoUtil.reescribir(self.ruta, [x.to_linea() for x in libros])
        return encontrado

# Prueba Bloque B

In [ ]:
# PRUEBA BLOQUE B — Repositorios (con archivos temporales)
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    ru = os.path.join(tmp, 'usuarios.txt')
    rc = os.path.join(tmp, 'categorias.txt')
    rl = os.path.join(tmp, 'libros.txt')
    repo_u, repo_c, repo_l = UsuarioRepository(ru), CategoriaRepository(rc), LibroRepository(rl)

    # Archivos inexistentes -> listas vacías
    assert repo_u.find_all() == [] and repo_l.find_all() == []
    print('✔ Archivos vacíos devuelven lista vacía')

    # Usuarios
    repo_u.guardar(Usuario('111', 'Ana Ruiz', TipoUsuarioEnum.DOCENTE, 'ana@c.com', '300'))
    repo_u.guardar(Usuario('222', 'Luis Mora', TipoUsuarioEnum.EXTERNO, 'luis@c.com', '310'))
    assert len(repo_u.find_all()) == 2
    assert repo_u.find_by_documento('111').nombre == 'Ana Ruiz'
    assert repo_u.find_by_documento('999') is None
    print('✔ Usuario: guardar y find_by_documento')

    u = repo_u.find_by_documento('222')
    u.estado = 'SUSPENDIDO'
    assert repo_u.actualizar(u) is True
    assert repo_u.find_by_documento('222').estado == 'SUSPENDIDO'
    assert repo_u.find_by_documento('111').estado == 'ACTIVO'
    assert len(repo_u.find_all()) == 2
    print('✔ Usuario: actualizar cambia solo ese registro')

    # Categorías
    repo_c.guardar(Categoria('LIT', 'Literatura', 'Novelas y cuentos'))
    repo_c.guardar(Categoria('TEC', 'Tecnología'))
    assert repo_c.find_by_codigo('TEC').descripcion is None
    assert repo_c.find_by_codigo('XXX') is None
    print('✔ Categoría: guardar y find_by_codigo')

    # Libros
    repo_l.guardar(Libro('978A', 'Cien anios de soledad', 'García Márquez', 'LIT', 1967, 5))
    repo_l.guardar(Libro('978B', 'Estructuras de datos en Python', 'Ana Gomez', 'TEC', 2020, 2))
    assert repo_l.find_by_isbn('978A').disponibles == 5
    assert [x.isbn for x in repo_l.find_by_titulo('PYTHON')] == ['978B']
    assert [x.isbn for x in repo_l.find_by_categoria('LIT')] == ['978A']
    print('✔ Libro: find_by_isbn, find_by_titulo, find_by_categoria')

    l = repo_l.find_by_isbn('978A')
    l.disponibles = 4
    assert repo_l.actualizar(l) is True
    assert repo_l.find_by_isbn('978A').disponibles == 4
    assert repo_l.actualizar(Libro('NOEXISTE', 'x', 'y', 'LIT', 2000, 1)) is False
    print('✔ Libro: actualizar (y False si no existe)')

print('\n✔ Bloque B verificado')

# Sección 3 — Servicios (P1)

In [ ]:
# ════════════════════════════════════
# CELDA 3.1 — UsuarioService
# ════════════════════════════════════
import re

ESTADOS_USUARIO = ('ACTIVO', 'SUSPENDIDO')


class UsuarioService:
    """Reglas de negocio para la gestión de usuarios (RF-01)."""

    def __init__(self, repo_usuarios):
        """Recibe el UsuarioRepository que usará para guardar y consultar."""
        self.repo = repo_usuarios

    def registrar(self, num_documento, nombre, tipo, correo, telefono):
        """Registra un usuario nuevo con estado ACTIVO.

        tipo debe ser un TipoUsuarioEnum. Lanza ValueError si algún dato es
        inválido o si el documento ya está registrado. Retorna el Usuario creado.
        """
        num_documento = num_documento.strip()
        nombre = nombre.strip()
        correo = correo.strip()
        telefono = telefono.strip()

        if not num_documento.isdigit():
            raise ValueError('El documento debe contener solo números.')
        if not nombre:
            raise ValueError('El nombre no puede estar vacío.')
        if not isinstance(tipo, TipoUsuarioEnum):
            raise ValueError('El tipo de usuario no es válido.')
        if not re.fullmatch(r'[^@\s|]+@[^@\s|]+\.[^@\s|]+', correo):
            raise ValueError('El correo no tiene un formato válido.')
        if not telefono.isdigit():
            raise ValueError('El teléfono debe contener solo números.')
        if self.repo.find_by_documento(num_documento) is not None:
            raise ValueError(f'Ya existe un usuario con el documento {num_documento}.')

        usuario = Usuario(num_documento, nombre, tipo, correo, telefono, 'ACTIVO')
        self.repo.guardar(usuario)
        return usuario

    def buscar_por_documento(self, doc):
        """Retorna el Usuario con ese documento, o None si no existe."""
        return self.repo.find_by_documento(doc.strip())

    def actualizar(self, doc, correo=None, telefono=None, estado=None):
        """Actualiza correo, teléfono y/o estado de un usuario existente.

        Solo cambia los datos que se indiquen (los que sean None se conservan).
        Lanza ValueError si el usuario no existe o algún dato es inválido.
        Retorna el Usuario actualizado.
        """
        usuario = self.repo.find_by_documento(doc.strip())
        if usuario is None:
            raise ValueError(f'No existe un usuario con el documento {doc}.')

        if correo is not None:
            correo = correo.strip()
            if not re.fullmatch(r'[^@\s|]+@[^@\s|]+\.[^@\s|]+', correo):
                raise ValueError('El correo no tiene un formato válido.')
            usuario.correo = correo
        if telefono is not None:
            telefono = telefono.strip()
            if not telefono.isdigit():
                raise ValueError('El teléfono debe contener solo números.')
            usuario.telefono = telefono
        if estado is not None:
            estado = estado.strip().upper()
            if estado not in ESTADOS_USUARIO:
                raise ValueError('El estado debe ser ACTIVO o SUSPENDIDO.')
            usuario.estado = estado

        self.repo.actualizar(usuario)
        return usuario

    def listar_todos(self):
        """Retorna la lista de todos los usuarios registrados."""
        return self.repo.find_all()
# ════════════════════════════════════
# CELDA 3.2 — LibroService
# ════════════════════════════════════
class LibroService:
    """Reglas de negocio para el catálogo de libros y categorías (RF-02)."""

    def __init__(self, repo_libros, repo_categorias):
        """Recibe el LibroRepository y el CategoriaRepository."""
        self.repo_libros = repo_libros
        self.repo_categorias = repo_categorias

    def registrar_categoria(self, codigo, nombre, descripcion=None):
        """Registra una categoría nueva.

        Lanza ValueError si faltan datos o el código ya existe.
        Retorna la Categoria creada.
        """
        codigo = codigo.strip().upper()
        nombre = nombre.strip()
        descripcion = descripcion.strip() if descripcion else None

        if not codigo:
            raise ValueError('El código de la categoría no puede estar vacío.')
        if not nombre:
            raise ValueError('El nombre de la categoría no puede estar vacío.')
        if self.repo_categorias.find_by_codigo(codigo) is not None:
            raise ValueError(f'Ya existe la categoría con código {codigo}.')

        categoria = Categoria(codigo, nombre, descripcion or None)
        self.repo_categorias.guardar(categoria)
        return categoria

    def listar_categorias(self):
        """Retorna la lista de todas las categorías registradas."""
        return self.repo_categorias.find_all()

    def registrar(self, isbn, titulo, autor, cod_categoria, anio, total_ejemplares):
        """Registra un libro nuevo con todos sus ejemplares disponibles.

        Lanza ValueError si algún dato es inválido, la categoría no existe
        o el ISBN ya está registrado. Retorna el Libro creado.
        """
        isbn = isbn.strip()
        titulo = titulo.strip()
        autor = autor.strip()
        cod_categoria = cod_categoria.strip().upper()

        if not isbn:
            raise ValueError('El ISBN no puede estar vacío.')
        if not titulo:
            raise ValueError('El título no puede estar vacío.')
        if not autor:
            raise ValueError('El autor no puede estar vacío.')
        if self.repo_categorias.find_by_codigo(cod_categoria) is None:
            raise ValueError(f'La categoría {cod_categoria} no existe. Regístrala primero.')
        if not isinstance(anio, int) or not (1000 <= anio <= 2100):
            raise ValueError('El año de publicación no es válido.')
        if not isinstance(total_ejemplares, int) or total_ejemplares < 1:
            raise ValueError('El número de ejemplares debe ser al menos 1.')
        if self.repo_libros.find_by_isbn(isbn) is not None:
            raise ValueError(f'Ya existe un libro con el ISBN {isbn}.')

        libro = Libro(isbn, titulo, autor, cod_categoria, anio, total_ejemplares)
        self.repo_libros.guardar(libro)
        return libro

    def buscar_por_isbn(self, isbn):
        """Retorna el Libro con ese ISBN, o None si no existe."""
        return self.repo_libros.find_by_isbn(isbn.strip())

    def buscar_por_titulo(self, txt):
        """Retorna los libros cuyo título contiene el texto indicado."""
        return self.repo_libros.find_by_titulo(txt.strip())

    def listar_por_categoria(self, cod):
        """Retorna los libros de una categoría."""
        return self.repo_libros.find_by_categoria(cod.strip().upper())

    def listar_disponibles(self):
        """Retorna los libros que tienen al menos un ejemplar disponible."""
        return [l for l in self.repo_libros.find_all() if l.disponibles > 0]

# Prueba Bloque C

In [ ]:
# PRUEBA BLOQUE C — Servicios (con archivos temporales)
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    repo_u = UsuarioRepository(os.path.join(tmp, 'usuarios.txt'))
    repo_c = CategoriaRepository(os.path.join(tmp, 'categorias.txt'))
    repo_l = LibroRepository(os.path.join(tmp, 'libros.txt'))
    srv_u = UsuarioService(repo_u)
    srv_l = LibroService(repo_l, repo_c)

    def debe_fallar(funcion, *args, **kwargs):
        """Retorna True si la función lanza ValueError."""
        try:
            funcion(*args, **kwargs)
            return False
        except ValueError:
            return True

    # --- Usuarios ---
    u = srv_u.registrar('1098765432', 'Maria Lopez', TipoUsuarioEnum.ESTUDIANTE,
                        'maria@correo.com', '3001234567')
    assert u.estado == 'ACTIVO'
    assert debe_fallar(srv_u.registrar, '1098765432', 'Otro', TipoUsuarioEnum.DOCENTE,
                       'o@c.com', '300')
    assert debe_fallar(srv_u.registrar, 'abc', 'X', TipoUsuarioEnum.DOCENTE, 'x@c.com', '300')
    assert debe_fallar(srv_u.registrar, '5', 'X', TipoUsuarioEnum.DOCENTE, 'sin-arroba', '300')
    assert debe_fallar(srv_u.registrar, '6', '   ', TipoUsuarioEnum.DOCENTE, 'x@c.com', '300')
    print('✔ Usuario: registrar y validaciones (duplicado, documento, correo, nombre)')

    assert srv_u.buscar_por_documento('1098765432').nombre == 'Maria Lopez'
    assert srv_u.buscar_por_documento('000') is None
    print('✔ Usuario: buscar_por_documento')

    srv_u.actualizar('1098765432', telefono='3119999999', estado='suspendido')
    u = srv_u.buscar_por_documento('1098765432')
    assert u.telefono == '3119999999' and u.estado == 'SUSPENDIDO'
    assert u.correo == 'maria@correo.com'
    assert debe_fallar(srv_u.actualizar, '1098765432', estado='BLOQUEADO')
    assert debe_fallar(srv_u.actualizar, '000', telefono='300')
    assert len(srv_u.listar_todos()) == 1
    print('✔ Usuario: actualizar contacto y estado, y listar_todos')

    # --- Categorías ---
    srv_l.registrar_categoria('lit', 'Literatura', 'Novelas')
    srv_l.registrar_categoria('TEC', 'Tecnología')
    assert srv_l.repo_categorias.find_by_codigo('LIT').nombre == 'Literatura'
    assert debe_fallar(srv_l.registrar_categoria, 'LIT', 'Repetida')
    assert debe_fallar(srv_l.registrar_categoria, '', 'Sin código')
    assert len(srv_l.listar_categorias()) == 2
    print('✔ Categoría: registrar (código en mayúsculas), duplicado y vacío rechazados')

    # --- Libros ---
    srv_l.registrar('978A', 'Cien anios de soledad', 'García Márquez', 'LIT', 1967, 5)
    srv_l.registrar('978B', 'Estructuras de datos en Python', 'Ana Gomez', 'TEC', 2020, 1)
    assert srv_l.buscar_por_isbn('978A').disponibles == 5
    assert debe_fallar(srv_l.registrar, '978A', 'Repetido', 'X', 'LIT', 2000, 1)
    assert debe_fallar(srv_l.registrar, '978C', 'Sin cat', 'X', 'ZZZ', 2000, 1)
    assert debe_fallar(srv_l.registrar, '978C', 'Sin copias', 'X', 'LIT', 2000, 0)
    assert debe_fallar(srv_l.registrar, '978C', 'Año malo', 'X', 'LIT', 20, 1)
    print('✔ Libro: registrar y validaciones (ISBN repetido, categoría, ejemplares, año)')

    assert [l.isbn for l in srv_l.buscar_por_titulo('python')] == ['978B']
    assert [l.isbn for l in srv_l.listar_por_categoria('lit')] == ['978A']
    print('✔ Libro: buscar por título y listar por categoría')

    libro = srv_l.buscar_por_isbn('978B')
    libro.disponibles = 0
    repo_l.actualizar(libro)
    assert [l.isbn for l in srv_l.listar_disponibles()] == ['978A']
    print('✔ Libro: listar_disponibles ignora los agotados')

print('\n✔ Bloque C verificado')

# Celda 4.0 — Funciones auxiliares de consola

In [ ]:
# ════════════════════════════════════
# CELDA 4.0 — Funciones auxiliares de consola
# ════════════════════════════════════
from datetime import datetime


def limpiar_pantalla():
    """Separa visualmente las pantallas imprimiendo líneas en blanco.

    En Colab no se usa os.system('clear'), porque no limpia la salida.
    """
    print('\n' * 2)


def leer_texto(mensaje, obligatorio=True):
    """Pide un texto por consola.

    Si es obligatorio, repite la pregunta hasta que no esté vacío.
    Si no lo es, retorna '' cuando el usuario solo presiona Enter.
    """
    while True:
        valor = input(mensaje).strip()
        if valor or not obligatorio:
            return valor
        print('  ✘ Este dato es obligatorio.')


def leer_entero(mensaje, minimo=None, maximo=None):
    """Pide un número entero por consola.

    Repite la pregunta si no es un entero o está fuera del rango indicado.
    """
    while True:
        valor = input(mensaje).strip()
        try:
            numero = int(valor)
        except ValueError:
            print('  ✘ Escribe un número entero.')
            continue
        if minimo is not None and numero < minimo:
            print(f'  ✘ El valor mínimo es {minimo}.')
            continue
        if maximo is not None and numero > maximo:
            print(f'  ✘ El valor máximo es {maximo}.')
            continue
        return numero


def leer_fecha(mensaje):
    """Pide una fecha en formato AAAA-MM-DD y la retorna como texto.

    Repite la pregunta hasta que la fecha sea válida.
    """
    while True:
        valor = input(mensaje).strip()
        try:
            datetime.strptime(valor, '%Y-%m-%d')
            return valor
        except ValueError:
            print('  ✘ Fecha inválida. Usa el formato AAAA-MM-DD (ej. 2025-10-15).')

# Celda 4.1 — menu_usuarios()

In [ ]:
# ════════════════════════════════════
# CELDA 4.1 — menu_usuarios()
# ════════════════════════════════════
def _elegir_tipo_usuario():
    """Pide al usuario que elija un tipo y retorna el TipoUsuarioEnum."""
    print('  Tipo de usuario:')
    print('    1. Estudiante')
    print('    2. Docente')
    print('    3. Externo')
    opcion = leer_entero('  Elige (1-3): ', 1, 3)
    return [TipoUsuarioEnum.ESTUDIANTE, TipoUsuarioEnum.DOCENTE, TipoUsuarioEnum.EXTERNO][opcion - 1]


def menu_usuarios(srv_usuarios):
    """Menú de gestión de usuarios (RF-01). Recibe el UsuarioService."""
    while True:
        limpiar_pantalla()
        print('═══ GESTIÓN DE USUARIOS ═══')
        print('1. Registrar usuario')
        print('2. Consultar usuario por documento')
        print('3. Actualizar contacto y estado')
        print('4. Listar todos los usuarios')
        print('0. Volver')
        opcion = input('Opción: ').strip()

        try:
            if opcion == '1':
                doc = leer_texto('Documento: ')
                nombre = leer_texto('Nombre completo: ')
                tipo = _elegir_tipo_usuario()
                correo = leer_texto('Correo: ')
                telefono = leer_texto('Teléfono: ')
                u = srv_usuarios.registrar(doc, nombre, tipo, correo, telefono)
                print(f'  ✔ Usuario registrado: {u}')

            elif opcion == '2':
                u = srv_usuarios.buscar_por_documento(leer_texto('Documento: '))
                print(f'  {u}' if u else '  ✘ No existe un usuario con ese documento.')

            elif opcion == '3':
                doc = leer_texto('Documento del usuario: ')
                print('  (Enter deja el dato sin cambios)')
                correo = leer_texto('  Nuevo correo: ', obligatorio=False) or None
                telefono = leer_texto('  Nuevo teléfono: ', obligatorio=False) or None
                estado = leer_texto('  Nuevo estado (ACTIVO/SUSPENDIDO): ', obligatorio=False) or None
                u = srv_usuarios.actualizar(doc, correo, telefono, estado)
                print(f'  ✔ Usuario actualizado: {u}')

            elif opcion == '4':
                usuarios = srv_usuarios.listar_todos()
                if not usuarios:
                    print('  (No hay usuarios registrados)')
                for u in usuarios:
                    print(f'  {u}')

            elif opcion == '0':
                return
            else:
                print('  ✘ Opción no válida.')
        except ValueError as e:
            print(f'  ✘ {e}')

# Celda 4.2 — menu_libros()

In [ ]:
# ════════════════════════════════════
# CELDA 4.2 — menu_libros()
# ════════════════════════════════════
def _mostrar_libros(libros):
    """Imprime una lista de libros, o un aviso si está vacía."""
    if not libros:
        print('  (No se encontraron libros)')
    for l in libros:
        print(f'  {l}')


def menu_libros(srv_libros):
    """Menú de gestión de libros y categorías (RF-02). Recibe el LibroService."""
    while True:
        limpiar_pantalla()
        print('═══ GESTIÓN DE LIBROS Y CATEGORÍAS ═══')
        print('1. Registrar categoría')
        print('2. Listar categorías')
        print('3. Registrar libro')
        print('4. Consultar libro por ISBN')
        print('5. Buscar libros por título')
        print('6. Listar libros por categoría')
        print('7. Listar libros con ejemplares disponibles')
        print('0. Volver')
        opcion = input('Opción: ').strip()

        try:
            if opcion == '1':
                codigo = leer_texto('Código (ej. LIT): ')
                nombre = leer_texto('Nombre: ')
                descripcion = leer_texto('Descripción (opcional): ', obligatorio=False)
                c = srv_libros.registrar_categoria(codigo, nombre, descripcion)
                print(f'  ✔ Categoría registrada: {c}')

            elif opcion == '2':
                categorias = srv_libros.listar_categorias()
                if not categorias:
                    print('  (No hay categorías registradas)')
                for c in categorias:
                    print(f'  {c}')

            elif opcion == '3':
                isbn = leer_texto('ISBN: ')
                titulo = leer_texto('Título: ')
                autor = leer_texto('Autor: ')
                cod_cat = leer_texto('Código de categoría: ')
                anio = leer_entero('Año de publicación: ')
                total = leer_entero('Número de ejemplares: ', minimo=1)
                l = srv_libros.registrar(isbn, titulo, autor, cod_cat, anio, total)
                print(f'  ✔ Libro registrado: {l}')

            elif opcion == '4':
                l = srv_libros.buscar_por_isbn(leer_texto('ISBN: '))
                print(f'  {l}' if l else '  ✘ No existe un libro con ese ISBN.')

            elif opcion == '5':
                _mostrar_libros(srv_libros.buscar_por_titulo(leer_texto('Texto del título: ')))

            elif opcion == '6':
                _mostrar_libros(srv_libros.listar_por_categoria(leer_texto('Código de categoría: ')))

            elif opcion == '7':
                _mostrar_libros(srv_libros.listar_disponibles())

            elif opcion == '0':
                return
            else:
                print('  ✘ Opción no válida.')
        except ValueError as e:
            print(f'  ✘ {e}')

# Prueba Bloque D

In [ ]:
# PRUEBA BLOQUE D — Menús (simulando input() con archivos temporales)
import tempfile
import builtins
import io
from contextlib import redirect_stdout

def simular(menu, servicio, entradas):
    """Ejecuta un menú dando las respuestas de 'entradas' y retorna lo impreso."""
    respuestas = iter(entradas)
    original = builtins.input
    builtins.input = lambda prompt='': next(respuestas)
    salida = io.StringIO()
    try:
        with redirect_stdout(salida):
            menu(servicio)
    finally:
        builtins.input = original
    return salida.getvalue()

with tempfile.TemporaryDirectory() as tmp:
    srv_u = UsuarioService(UsuarioRepository(os.path.join(tmp, 'u.txt')))
    srv_l = LibroService(LibroRepository(os.path.join(tmp, 'l.txt')),
                         CategoriaRepository(os.path.join(tmp, 'c.txt')))

    # Usuarios: registrar (tipo 2 = Docente), duplicado, consultar, listar, salir
    out = simular(menu_usuarios, srv_u, [
        '1', '111', 'Ana Ruiz', '2', 'ana@c.com', '300',
        '1', '111', 'Otra', '1', 'o@c.com', '300',
        '2', '111',
        '4',
        '0'])
    assert '✔ Usuario registrado' in out
    assert 'Ya existe un usuario' in out
    assert 'Ana Ruiz' in out and 'DOCENTE' in out
    print('✔ menu_usuarios: registrar, rechazar duplicado, consultar y listar')

    # Libros: categoría, libro con año no numérico (reintenta), consulta, salir
    out = simular(menu_libros, srv_l, [
        '1', 'lit', 'Literatura', '',
        '3', '978A', 'Cien anios', 'García Márquez', 'LIT', 'abc', '1967', '5',
        '4', '978A',
        '7',
        '0'])
    assert '✔ Categoría registrada' in out
    assert 'Escribe un número entero' in out
    assert '✔ Libro registrado' in out
    assert '5/5 disponibles' in out
    print('✔ menu_libros: categoría, libro (con reintento de año) y consultas')

print('\n✔ Bloque D verificado')








# Sección 5 — Ejecución (P1: menú provisional)

In [ ]:
# ════════════════════════════════════
# CELDA 5.1 — Ejecución (menú provisional de P1)
# ════════════════════════════════════
# Se crean los repositorios y servicios con las rutas reales de la Celda 0.
repo_usuarios = UsuarioRepository(RUTA_USUARIOS)
repo_categorias = CategoriaRepository(RUTA_CATEGORIAS)
repo_libros = LibroRepository(RUTA_LIBROS)

srv_usuarios = UsuarioService(repo_usuarios)
srv_libros = LibroService(repo_libros, repo_categorias)


def menu_principal():
    """Menú principal provisional de P1 (usuarios y libros)."""
    while True:
        print('\n╔══════════════════════════════╗')
        print('║  SISTEMA DE GESTIÓN BIBLIOTECARIA  ║')
        print('╚══════════════════════════════╝')
        print('1. Usuarios')
        print('2. Libros y categorías')
        print('0. Salir')
        opcion = input('Opción: ').strip()

        if opcion == '1':
            menu_usuarios(srv_usuarios)
        elif opcion == '2':
            menu_libros(srv_libros)
        elif opcion == '0':
            print('Hasta pronto.')
            return
        else:
            print('  ✘ Opción no válida.')


menu_principal()